In [1]:
import json
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from helpers.triplets import TripletGenerator, transform_triplets_for_embedding, triplet_to_natural_language
from llama_index.llms.openai import OpenAI
from llama_index.core import SimpleDirectoryReader, KnowledgeGraphIndex
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import Settings
from IPython.display import Markdown, display
from dotenv import load_dotenv
from llama_index.core import StorageContext
from llama_index.core.schema import TextNode
from typing import List, Tuple, Dict
import os
from tqdm import tqdm
from datetime import datetime
import re

load_dotenv("devops/env/default.env")
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

with open('btc_blocks.json', 'r') as f:
    blocks_data = json.load(f)

with open('economic_indicators.json', 'r') as f:
    economic_data = json.load(f)

with open('on_chain_metrics.json', 'r') as f:
    onchain_data = json.load(f)


tg = TripletGenerator()
triplets = tg.load_and_process_data(blocks_data, economic_data, onchain_data)

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = OpenAI(model="gpt-3.5-turbo-instruct", 
             temperature=0,
             api_key=OPENAI_API_KEY)
Settings.llm = llm
Settings.embed_model = embed_model

graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

kg_index = KnowledgeGraphIndex(
    [],
    storage_context=storage_context,
    include_embeddings=True,
)


# The existing upsert_triplets performs sequential insertion including sequential embedding generation which is verrryyy slow
def batch_upsert_triplets_and_nodes(kg: KnowledgeGraphIndex, triplets: List[Tuple[Tuple[str, str, str], Dict]], include_embeddings: bool = True) -> None:
    """Batch insert triplets with their embeddings."""
    # First insert all triplets to the graph store
    for (subj, pred, obj), metadata in triplets:
        # Add the triplet to the graph store
        kg._graph_store.upsert_triplet(subj, pred, obj)

        # Create a node that represents this triplet with its metadata
        triplet_text = triplet_to_natural_language((subj, pred, obj))
        node_id = f"{subj}_{pred}_{obj}"
        
        # Create a node with the metadata
        triplet_node = TextNode(
            text=triplet_text,
            metadata=metadata,
            id_=node_id
        )
        kg.add_node([subj, obj], triplet_node)
    
    if include_embeddings:
        # Generate triplet strings for embedding
        triplet_strs = transform_triplets_for_embedding([triplet for triplet, _ in triplets])
        
        # Get embeddings in batch
        batch_embeddings = kg._embed_model.get_text_embedding_batch(triplet_strs)
        
        # Add embeddings to the index structure
        for triplet_str, embedding in tqdm(zip(triplet_strs, batch_embeddings)):
            kg._index_struct.add_to_embedding_dict(triplet_str, embedding)
            
        # Update the storage
        kg._storage_context.index_store.add_index_struct(kg._index_struct)



#batch_upsert_triplets_and_nodes(kg_index, triplets)


/tmp/ipykernel_10939/3911633747.py:44: DeprecationWarning: Call to deprecated class KnowledgeGraphIndex. (The KnowledgeGraphIndex class has been deprecated. Please use the new PropertyGraphIndex class instead. If a certain graph store integration is missing in the new class, please open an issue on the GitHub repository or contribute it!) -- Deprecated since version 0.10.53.
  kg_index = KnowledgeGraphIndex(


In [5]:
for i in range(len(triplets)):
    if i % 1000 == 0:
        print(triplets[i])

(('Block:894214', 'HAS_HASH', '00000000000000000000ab2d5af936a7c0b91f4216ac3a3e7f0ba89d9f20e03f'), {'year': None, 'month': None, 'day': None, 'hour': None, 'metric_type': None, 'indicator_type': None, 'block_height': '894214', 'txid': None})
(('OnChainMetric:transaction_volume_usd', 'HAS_VALUE_AT', 'Date:2025-04-18'), {'year': 2025, 'month': 4, 'day': 18, 'hour': 0, 'metric_type': 'transaction_volume_usd', 'indicator_type': None, 'block_height': None, 'txid': None})
(('OnChainMetric:mempool_size: Date:2025-03-31', 'HAS_VALUE', '6381844.5'), {'year': 2025, 'month': 3, 'day': 31, 'hour': 0, 'metric_type': 'mempool_size', 'indicator_type': None, 'block_height': None, 'txid': None})
(('OnChainMetric:mempool_size: Date:2025-04-07', 'MEASURED_AT', 'Date:2025-04-07'), {'year': 2025, 'month': 4, 'day': 7, 'hour': 0, 'metric_type': 'mempool_size', 'indicator_type': None, 'block_height': None, 'txid': None})
(('OnChainMetric:mempool_size', 'HAS_VALUE_AT', 'Date:2025-04-14'), {'year': 2025, 'mont

In [5]:
query_engine = kg_index.as_query_engine(
    similarity_top_k=10,
    embedding_mode="hybrid",
    response_mode="tree_summarize", #tree_summarize, no_text
    include_text=False,
)

In [3]:
query_engine.query("When was block 000000000000000000023175d87f3d7270a30fb4b515ea06dcb045d46f4d78ab created?")

Response(response='2025-04-27', source_nodes=[NodeWithScore(node=TextNode(id_='c82d7321-de05-4a8d-8102-b6d7cf50f137', embedding=None, metadata={'kg_rel_texts': ['Beb61781167390F57708Bb437A84E8C0B0Ad53Bff421Ac4E3Dd5C72160349958 belongs to 894214.', '0F624D1677F0Eda50De267E8F23Ceea263E9B295C5991Da3D854D1388Ef4820B belongs to 894214.', 'Sp500 Bitcoin Hash Rate:2025-04-07 occurred on 2025-04-07.', 'Sp500 Bitcoin Hash Rate:2025-04-14 occurred on 2025-04-14.', 'Sp500 Bitcoin Hash Rate:2025-04-09 occurred on 2025-04-09.', 'Sp500 Bitcoin Hash Rate:2025-04-08 occurred on 2025-04-08.', '894214 was created on Timestamp:2025-04-27 20:05:04.', '894214 was created on Date:2025-04-27.', '894214 occurred on 2025-04-27.', '2025-04-27 has block 894214.'], 'kg_rel_map': {'block': [], '000000000000000000023175d87f3d7270a30fb4b515ea06dcb045d46f4d78ab': [], 'created': []}}, excluded_embed_metadata_keys=['kg_rel_map', 'kg_rel_texts'], excluded_llm_metadata_keys=['kg_rel_map', 'kg_rel_texts'], relationships={

In [4]:
query_engine.query("What was the Bitcoin Transaction Volume on 25th April 2025")

Response(response='\nThe Bitcoin Transaction Volume on 25th April 2025 was 17332578422.84961 USD and 392451 BTC.', source_nodes=[NodeWithScore(node=TextNode(id_='995ef7d2-4f23-43ec-aa93-5c88ad8ed616', embedding=None, metadata={'kg_rel_texts': ['The 23Rd April 2025 has a bitcoin transaction volume usd of 16200026210.79612.', 'The 25Th April 2025 has a bitcoin transaction volume usd of 17332578422.84961.', 'The 26Th April 2025 has a bitcoin transaction volume usd of 4555887933.739448.', 'The 20Th April 2025 has a bitcoin transaction volume usd of 2232630980.687477.', 'The 24Th April 2025 has a bitcoin transaction volume btc of 464739.0.', 'The 19Th April 2025 has a bitcoin transaction volume btc of 545076.0.', 'The 17Th April 2025 has a bitcoin transaction volume btc of 557181.0.', 'The 25Th April 2025 has a bitcoin transaction volume btc of 392451.0.', 'The 20Th April 2025 has a bitcoin transaction volume btc of 468737.0.', 'The 26Th April 2025 has a bitcoin transaction volume btc of 32

In [12]:
query_engine.query("How has the mining difficulty changed over recent blocks")

Response(response='The mining difficulty has increased over recent blocks.', source_nodes=[NodeWithScore(node=TextNode(id_='424f92f9-c79a-41ac-84c7-53d9c7ee8d5f', embedding=None, metadata={'kg_rel_texts': ['Difficulty is described as A Relative Measure Of How Difficult It Is To Find A New Block. The Difficulty Is Adjusted Periodically As A Function Of How Much Hashing Power Has Been Deployed By The Network Of Miners..', 'The Sp500 Difficulty Metric:2025-04-25 has a indicator value of 5525.21.', 'Difficulty is displayed as Bitcoin Network Difficulty.', 'Bitcoin Network Difficulty correlates with Sp500.', '2025-04-25 has difficulty metric Difficulty.', 'Bitcoin Difficulty correlates with Sp500.', 'Difficulty is measured in Difficulty.', 'Difficulty has value at 1745539200.', 'Difficulty has value on 2025-04-25.', 'Difficulty has period Day.'], 'kg_rel_map': {'mining': [], 'difficulty': [], 'recent': [], 'blocks': [], 'changed': []}}, excluded_embed_metadata_keys=['kg_rel_map', 'kg_rel_te

In [11]:
from llama_index.core.vector_stores import FilterOperator, MetadataFilter, MetadataFilters

filters = []

#filters.append(MetadataFilter(key="block_height", operator=FilterOperator.EQ, value=894214))
filters.append(MetadataFilter(key="metric_type", operator=FilterOperator.EQ, value="transaction_volume_btc"))

# Create the metadata filters object if we have any filters
metadata_filters = MetadataFilters(filters=filters) if filters else None


# Execute the query with filters
retriever = kg_index.as_retriever(filters=metadata_filters, similarity_top_k=10)
nodes = retriever.retrieve("")

In [12]:
for node, score in nodes:
    print(node[1].metadata)

{'kg_rel_texts': ['Dollar Index Indicator Bitcoin Active Addresses:2025-04-15 occurred on 2025-04-15.', 'Dollar Index Indicator Bitcoin Active Addresses:2025-04-14 occurred on 2025-04-14.', 'Dollar Index Indicator Bitcoin Active Addresses:2025-04-17 occurred on 2025-04-17.', 'Sp500 Indicator Bitcoin Active Addresses:2025-04-14 occurred on 2025-04-14.', 'Sp500 Indicator Bitcoin Active Addresses:2025-04-17 occurred on 2025-04-17.', 'Sp500 Indicator Bitcoin Active Addresses:2025-04-15 occurred on 2025-04-15.', 'Dollar Index Bitcoin Active Addresses:2025-04-14 occurred on 2025-04-14.', 'Dollar Index Bitcoin Active Addresses:2025-04-17 occurred on 2025-04-17.', 'Dollar Index Bitcoin Active Addresses:2025-04-15 occurred on 2025-04-15.', 'Active Addresses is displayed as Bitcoin Active Addresses.'], 'kg_rel_map': {'answers': [], 'question': [], 'stopwords': [], 'keywords': [], 'extract': [], '10': [], 'lookup': [], 'text': []}}
